# 全流程跑通（Byte-Session 主链路）

## 目标
从数据会话构建到训练、离线评估、实时推理、API 入库的整条链路可运行。

## 阶段 1：会话样本构建
在 `costSensitive` 目录执行：

```powershell
python session_data.py --dataset-root dataset/iscx --output-root processed_full/sessions --num-packets 12 --packet-len 256 --train-ratio 0.8 --flow-timeout-s 10 --seed 42
```

检查产物：
- `processed_full/sessions/manifest.csv`
- `processed_full/sessions/label_map.json`
- `processed_full/sessions/summary.json`
- `processed_full/sessions/samples/*.npz`

## 阶段 2：训练

```powershell
python train_byte_session.py --manifest processed_full/sessions/manifest.csv --max_epoch 8 --batch-size 32
```

检查产物：
- `pytorch_model/byte_session_classifier.pth`
- `pytorch_model/byte_session_train_log.csv`
- `pytorch_model/byte_session_train_report.json`

## 阶段 3：离线推理与评估

```powershell
python inference.py --manifest processed_full/sessions/manifest.csv --model-path pytorch_model/byte_session_classifier.pth --split test
```

检查产物：
- `pytorch_model/session_predictions.csv`
- `pytorch_model/session_embeddings.npy`
- `pytorch_model/session_eval_report.json`

## 阶段 4：未知类检测器（可选但推荐）

```powershell
python build_centroid_detector.py --manifest processed_full/sessions/manifest.csv --model-path pytorch_model/byte_session_classifier.pth --output-json pytorch_model/centroid_detector.json
```

## 阶段 5：实时推理
当前仅支持 `live`：

```powershell
python main_realtime.py --mode live --source "\\Device\\NPF_{你的网卡GUID}" --model-path pytorch_model/byte_session_classifier.pth --label-map processed_full/sessions/label_map.json --detector-json pytorch_model/centroid_detector.json --output-dir pytorch_model/realtime_sessions
```

检查产物：
- `pytorch_model/realtime_sessions/realtime_predictions.csv`
- `pytorch_model/realtime_sessions/realtime_predictions.jsonl`

## 阶段 6：联动 API + 入库引擎
回到仓库根目录执行：

```powershell
python run_full_chain.py --host 127.0.0.1 --port 8000 --reload
```

默认会自动拉起：
- FastAPI 服务（`services/api_server.py`）
- SQLite 入库引擎（`services/backend_engine.py`）
- 可选实时抓包子进程（受环境变量控制）